In [ ]:
dataset_url = ""
dataset_base64 = ""
target_column = ""
n_estimators = 100
test_size = 0.2

In [ ]:
import pandas as pd
import numpy as np
import json, io, base64, warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, r2_score, mean_squared_error,
                              mean_absolute_error, confusion_matrix)

result = {}
try:
    # ── Load data ──────────────────────────────────────────────────────────────
    if dataset_url:
        ext = dataset_url.split('.')[-1].lower()
        df = (pd.read_excel(dataset_url) if ext in ['xlsx', 'xls']
              else pd.read_json(dataset_url) if ext == 'json'
              else pd.read_csv(dataset_url))
    elif dataset_base64:
        raw = base64.b64decode(dataset_base64)
        try:
            df = pd.read_csv(io.BytesIO(raw))
        except Exception:
            df = pd.read_excel(io.BytesIO(raw))
    else:
        from sklearn.datasets import load_iris
        iris = load_iris()
        df = pd.DataFrame(iris.data, columns=iris.feature_names)
        df['target'] = iris.target
        if not target_column:
            target_column = 'target'

    # ── Detect target ──────────────────────────────────────────────────────────
    tgt = target_column if target_column else df.columns[-1]
    if tgt not in df.columns:
        raise ValueError(f"Target column '{tgt}' not found. Available: {list(df.columns)}")

    # ── Preprocessing ──────────────────────────────────────────────────────────
    df = df.dropna(subset=[tgt]).copy()
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
            df[col] = df[col].fillna(df[col].median())
        else:
            df[col] = df[col].fillna(df[col].mode()[0] if not df[col].mode().empty else 'unknown')
    for col in df.select_dtypes(include='object').columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))

    X = df.drop(columns=[tgt])
    y = df[tgt]
    is_clf = (y.nunique() < 10) or str(y.dtype) in ['object', 'bool', 'category']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=float(test_size), random_state=42, stratify=y if is_clf else None)

    # ── Train ──────────────────────────────────────────────────────────────────
    if is_clf:
        model = RandomForestClassifier(n_estimators=int(n_estimators), random_state=42)
    else:
        model = RandomForestRegressor(n_estimators=int(n_estimators), random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # ── Metrics ────────────────────────────────────────────────────────────────
    result = {
        'algorithm': 'random_forest',
        'task_type': 'classification' if is_clf else 'regression',
        'metrics': {}
    }
    if is_clf:
        result['metrics']['accuracy']  = round(float(accuracy_score(y_test, y_pred)), 4)
        result['metrics']['precision'] = round(float(precision_score(y_test, y_pred, average='weighted', zero_division=0)), 4)
        result['metrics']['recall']    = round(float(recall_score(y_test, y_pred, average='weighted', zero_division=0)), 4)
        result['metrics']['f1_score']  = round(float(f1_score(y_test, y_pred, average='weighted', zero_division=0)), 4)
        result['confusion_matrix'] = confusion_matrix(y_test, y_pred).tolist()
    else:
        result['metrics']['r2_score'] = round(float(r2_score(y_test, y_pred)), 4)
        result['metrics']['rmse']     = round(float(np.sqrt(mean_squared_error(y_test, y_pred))), 4)
        result['metrics']['mae']      = round(float(mean_absolute_error(y_test, y_pred)), 4)

    feat_imp = sorted(zip(X.columns.tolist(), model.feature_importances_.tolist()), key=lambda x: -x[1])
    result['feature_importance'] = [{'feature': f, 'importance': round(float(i), 4)} for f, i in feat_imp]

    n_s = min(20, len(y_test))
    if is_clf:
        result['sample_predictions'] = [
            {'actual': int(a), 'predicted': int(p)}
            for a, p in zip(y_test.values[:n_s], y_pred[:n_s])
        ]
    else:
        result['sample_predictions'] = [
            {'actual': round(float(a), 4), 'predicted': round(float(p), 4)}
            for a, p in zip(y_test.values[:n_s], y_pred[:n_s])
        ]

except Exception as e:
    import traceback
    result = {'error': str(e), 'traceback': traceback.format_exc()}

with open('/tmp/ml_result.json', 'w') as f:
    json.dump(result, f)
print(json.dumps(result, indent=2))